In [1]:
%pip install liftover

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from liftover import get_lifter

In [34]:
gwas = pd.read_csv(r'C:\Users\micha\OneDrive\Documents\PRSGWAS_tsv\EU_Data_Gigastroke_Cleaned.txt', sep='\t')
converter = get_lifter('hg19', 'hg38')

In [35]:
gwas

,CHR,BP,A1,A2,BETA,P,EAF,SE,OR,MAF
0,5,29439275,T,C,0.0069,0.36030,0.3566,0.0076,1.006924,0.3566
1,5,85928892,T,C,-0.0148,0.32290,0.0635,0.0150,0.985309,0.0635
2,10,128341232,T,C,0.0069,0.33070,0.4610,0.0071,1.006924,0.4610
3,3,62707519,T,C,0.0169,0.30460,0.0536,0.0164,1.017044,0.0536
4,2,80464120,T,G,-0.0043,0.87490,0.9785,0.0273,0.995709,0.0215
...,...,...,...,...,...,...,...,...,...,...
7482027,3,80436770,A,G,-0.0460,0.01529,0.9578,0.0190,0.955042,0.0422
7482028,18,44156714,C,G,0.0044,0.63330,0.1847,0.0093,1.004410,0.1847
7482029,5,95076854,A,G,-0.0179,0.19660,0.9176,0.0139,0.982259,0.0824
7482030,11,99622387,T,C,0.0127,0.15010,0.8017,0.0088,1.012781,0.1983


In [36]:
converter

ChainFile("C:\Users\micha/.liftover\hg19ToHg38.over.chain.gz")

In [37]:
def do_liftover(chrom, pos):
    # Liftover expects chromosome names to start with 'chr' (e.g., 'chr1' instead of '1')
    chr_str = str(chrom)
    if not chr_str.startswith('chr'):
        chr_str = 'chr' + chr_str
        
    # Query the liftover map
    new_coords = converter.query(chr_str, int(pos))
    
    if new_coords:
        # new_coords returns a list of matches. We take the position of the first match.
        return new_coords[0][1] 
    else:
        # If the position no longer exists in hg38, return None
        return None

In [38]:
gwas['hg38_pos'] = gwas.apply(lambda row: do_liftover(row['CHR'], row['BP']), axis=1)

In [39]:
# Drop SNPs that couldn't be mapped to hg38
gwas_clean = gwas.dropna(subset=['hg38_pos']).copy()

# Convert the new positions back to normal integers
gwas_clean['hg38_pos'] = gwas_clean['hg38_pos'].astype(int)

# Replace the old hg19 column with the new hg38 column
gwas_clean['BP'] = gwas_clean['hg38_pos']
gwas_clean = gwas_clean.drop(columns=['hg38_pos'])

In [40]:
gwas_clean

,CHR,BP,A1,A2,BETA,P,EAF,SE,OR,MAF
0,5,29439168,T,C,0.0069,0.36030,0.3566,0.0076,1.006924,0.3566
1,5,86633075,T,C,-0.0148,0.32290,0.0635,0.0150,0.985309,0.0635
2,10,126652663,T,C,0.0069,0.33070,0.4610,0.0071,1.006924,0.4610
3,3,62721844,T,C,0.0169,0.30460,0.0536,0.0164,1.017044,0.0536
4,2,80236995,T,G,-0.0043,0.87490,0.9785,0.0273,0.995709,0.0215
...,...,...,...,...,...,...,...,...,...,...
7482027,3,80387620,A,G,-0.0460,0.01529,0.9578,0.0190,0.955042,0.0422
7482028,18,46576751,C,G,0.0044,0.63330,0.1847,0.0093,1.004410,0.1847
7482029,5,95741150,A,G,-0.0179,0.19660,0.9176,0.0139,0.982259,0.0824
7482030,11,99751656,T,C,0.0127,0.15010,0.8017,0.0088,1.012781,0.1983


In [41]:
gwas_clean.to_csv(r'C:\Users\micha\OneDrive\Documents\PRSGWAS_tsv\eu_gigastroke_hg38.txt', sep='\t', index=False)